# Problem Set 3
## Problem 1: Gibbs Sampling
### 1. Algorithm

In [1]:
import numpy as np

def gibbs(A, s, t, w, burnin, its):
    """Gibbs Sampling algorithm

    Arguments:
        A {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        w {n x n matrix} -- weights for each edge
        burnin {integer} -- number of burn-in samples
        its {integer} -- number of iterations after burn-in

    Returns:
        A vector of marginals whose ith component is equal to the probability
        that i ∈ V is in the same part of the cut as s.
    """

    n = A.shape[0]
    steps = burnin + its

    # Start with a valid sample
    sample = np.zeros(n)
    sample[s] = 1
    samples = np.empty((its, n))

    # We'll add any samples after burnin to totals
    totals = np.zeros(n)

    vertices = np.arange(n)
    # Don't consider changing s or t
    vertices = vertices[vertices != s]
    vertices = vertices[vertices != t]


    neighbors = [[idx for idx, edge in enumerate(A[i]) if edge == 1] for i in range(n)]

    for step in range(0, steps):
        # Randomize the order every time
        np.random.shuffle(vertices)

        for i in vertices:
            prob0 = 0
            prob1 = 0

            # Probability of x_i increases by w[i][j] whenever i != j
            for j in neighbors[i]:
                if sample[j] == 1:
                    prob0 += w[i][j]
                else:
                    prob1 += w[i][j]

            # Normalize
            prob1 = prob1 / (prob0 + prob1)

            # Stochastically update sample[i]
            # based on the marginal of i
            # that we calculated
            sample[i] = int(np.random.rand() < prob1)

        if step >= burnin:
            totals += sample
            samples[step - burnin] = sample

    return totals / its, samples

In [2]:
#              s, a, b, c, d, e, f, t
A = np.array([[0, 1, 1, 1, 0, 0, 0, 0],  # s
              [1, 0, 0, 1, 1, 0, 0, 0],  # a
              [1, 0, 0, 1, 0, 0, 0, 0],  # b
              [1, 1, 1, 0, 0, 1, 0, 0],  # c
              [0, 1, 0, 0, 0, 0, 1, 1],  # d
              [0, 0, 0, 1, 0, 0, 1, 1],  # e
              [0, 0, 0, 0, 1, 1, 0, 1],  # f
              [0, 0, 0, 0, 1, 1, 1, 0]], # t
              dtype = np.float64)

#              s, a, b, c, d, e, f, t
w = np.array([[0, 0.5, 0.5, 0.5, 0, 0, 0, 0],   # s
              [0.5, 0, 0, 0.5, 0.5, 0, 0, 0],   # a
              [0.5, 0, 0, 0.5, 0, 0, 0, 0],     # b
              [0.5, 0.5, 0.5, 0, 0, 0.5, 0, 0], # c
              [0, 0.5, 0, 0, 0, 0, 0.5, 0.5],   # d
              [0, 0, 0, 0.5, 0, 0, 0.5, 0.5],   # e
              [0, 0, 0, 0, 0.5, 0.5, 0, 0.5],   # f
              [0, 0, 0, 0, 0.5, 0.5, 0.5, 0]],  # t
              dtype = np.float64)

s = 0
t = 7
params = [2 ** 6, 2 ** 10, 2 ** 14, 2 ** 18]

print("Gibbs Sampling")
print("Estimates for the marginal of e:")
print()
print(f"{'Its':^50}")
print(f"{'Burnin':<8}|", *[f"{p:^8}" for p in params])
print("-" * 50)
for burnin in params:
    print(f"{burnin:<8}|", *[f"{gibbs(A, s, t, w, burnin, its)[0][5]:^8.2f}" for its in params])

Gibbs Sampling
Estimates for the marginal of e:

                       Its                        
Burnin  |    64      1024    16384    262144 
--------------------------------------------------
64      |   0.72     0.70     0.68     0.68  
1024    |   0.62     0.72     0.68     0.67  
16384   |   0.64     0.69     0.68     0.68  
262144  |   0.67     0.68     0.67     0.67  


## Problem 2: Maximum Likelihood for s-t Cuts 
### 1. Algorithm

In [3]:
# learnstcuts
from collections import defaultdict
from copy import deepcopy

import numpy as np

def learnstcuts(A, s, t, samples):
    """Maximum Likelihood for s-t Cuts

    Arguments:
        A {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        samples {n x m matrix} --
            A list of m samples from which we'll learn an n x n vector of weights

    Returns:
        An n x n-dimensional vector of weights w corresponding
        to the MLE parameters for the model.
    """

    n, m = samples.shape
    sample_probs = np.mean(samples, axis=1)

    phi = lambda i, x_i: sample_probs[i] if bool(x_i) else 1 - sample_probs[i]
    psi = lambda i, j, x_i, x_j: 1 + int(x_i != x_j)

    edges = []
    for i in range(n):
        for j in range(n):
            if A[i,j] == 1:
                edges.append((i,j))

    # Stores messages from factor nodes (C) to vertices (i).
    # C_i_messages[(C, i), i, x_i] represents the message
    # from the factor (C) to the vertex (i)
    C_i_messages = defaultdict(lambda: 1)

    # Stores messages from vertices (i) to factor nodes (C).
    # i_C_messages[i, (i, C), x_i] represents the message
    # from the vertex (i) to the factor (C)
    i_C_messages = defaultdict(lambda: 1)
    new_C_i_messages = defaultdict(lambda: 1)

    for _ in range(m):
        # Messages from C to i
        for (C, i) in edges:
            for x_i in range(2):
                C_i_max = 0

                # Find the max over assignments to x_C
                for x_C in range(2):
                    # Product over k in C \ i = {C}
                    # Would be a loop if I was using cliques and not just edges
                    C_i_max = max(C_i_max, psi(C, i, x_C, x_i) * i_C_messages[C, (C, i), x_C])

                new_C_i_messages[(C, i), i, x_i] = C_i_max

        # Messages from i to C
        for (i, C) in edges:
            for x_i in range(2):
                prod = phi(i, x_i)

                # Product over messages into i
                for (C_prime, k) in edges:
                    # Exclude C -> i
                    if k == i and C_prime != C:
                        prod *= C_i_messages[(C_prime, k), k, x_i]

                i_C_messages[i, (i, C), x_i] = prod

        C_i_messages = deepcopy(new_C_i_messages)

        # Scale
        max_val = max(map(abs, [*C_i_messages.values(), *i_C_messages.values()]))
        if max_val > 0:
            for i in C_i_messages:
                C_i_messages[i] = max(1e-50, C_i_messages[i] / max_val)
            for i in i_C_messages:
                i_C_messages[i] = max(1e-50, i_C_messages[i] / max_val)

    # Calculate beliefs for all i
    beliefs = np.empty(n)
    for i in range(n):
        belief0 = phi(i, 0)
        belief1 = phi(i, 1)

        for (C, k) in edges:
            if k == i:
                belief0 *= C_i_messages[(C, k), k, 0]
                belief1 *= C_i_messages[(C, k), k, 1]
        beliefs[i] = belief1 / (belief0 + belief1)

    # Calculate beliefs for all x_C
    C_beliefs = defaultdict(int)
    for (i, j) in edges:
        if ((j, i), (0, 0)) in C_beliefs:
            continue

        total = 0

        for x_i in range(2):
            for x_j in range(2):
                prob = psi(i, j, x_i, x_j)

                # Message i -> C (x_i)
                prob *= i_C_messages[i, (i, j), x_i]
                # Message j -> C (x_j)
                prob *= i_C_messages[j, (j, i), x_j]

                C_beliefs[(i, j), (x_i, x_j)] = prob
                total += prob

        # Normalize
        for x_i in range(2):
            for x_j in range(2):
                C_beliefs[(i, j), (x_i, x_j)] /= total

    # Construct w-matrix
    weights = np.zeros((n,n))
    for (i, j), (x_i, x_j) in C_beliefs:
        # We only need weights for x_C = 1
        # x_C is 1 where adjacent vertices are mismatched
        if x_i != x_j:
            # Make sure weights are mirrored
            weights[i, j] += C_beliefs[(i, j), (x_i, x_j)]
            weights[j, i] += C_beliefs[(i, j), (x_i, x_j)]

    return weights

In [6]:
import math

A = np.array([[0, 1, 0],  # 0
              [1, 0, 1],  # 1
              [0, 1, 0]], # 2
              dtype = np.float64)

w = np.array([[0, math.exp(1), 0],           # 0
              [math.exp(1), 0, math.exp(1)], # 1
              [0, math.exp(1), 0]],          # 2
              dtype = np.float64)

s = 0
t = 2
its = 1000
burnin = 1000

samples = gibbs(A, s, t, w, its, burnin)[1]

weights = learnstcuts(A, s, t, np.transpose(samples))

print("Approximate Weights:")
print(weights)

Approximate Weights:
[[0.    0.491 0.   ]
 [0.491 0.    0.509]
 [0.    0.509 0.   ]]


In [7]:
#              a, b, c, d
A = np.array([[0, 1, 1, 0],  # a
              [1, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # b
              dtype = np.float64)

#              a, b, c, d
w = np.array([[0, 2, 1, 0],  # a
              [2, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # d
              dtype = np.float64)

s = 0
t = 3
its = 1000
burnin = 1000

samples = gibbs(A, s, t, w, its, burnin)[1]

weights = learnstcuts(A, s, t, np.transpose(samples))

print("Approximate Weights:")
print(weights)

Approximate Weights:
[[0.    0.669 0.502 0.   ]
 [0.669 0.    0.    0.331]
 [0.502 0.    0.    0.498]
 [0.    0.331 0.498 0.   ]]


In [8]:
# Adjacency for a "star": 0 connected to 1,2,3
n = 4
A = np.zeros((n,n), dtype = np.float64)
for (i,j) in [(0,1), (0,2), (0,3)]:
    A[i,j] = 1
    A[j,i] = 1

# Let's do w=2 for edges (0,1) and (0,2), and w=3 for (0,3)
w = np.zeros((n,n), dtype = np.float64)
edges = [(0,1,2.0), (0,2,2.0), (0,3,3.0)]
for (i,j,wval) in edges:
    w[i,j] = wval
    w[j,i] = wval

s = 0
t = 3
its = 1000
burnin = 1000

samples = gibbs(A, s, t, w, its, burnin)[1]

weights = learnstcuts(A, s, t, np.transpose(samples))

print("Approximate Weights:")
print(weights)

Approximate Weights:
[[0. 1. 1. 1.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]]


In [9]:
A = np.array([
    [0,1,0,0,1],
    [1,0,1,0,0],
    [0,1,0,1,0],
    [0,0,1,0,1],
    [1,0,0,1,0]
], dtype = np.float64)

# Weights (w):
w = np.array([
    [0,2,0,0,4],
    [2,0,3,0,0],
    [0,3,0,5,0],
    [0,0,5,0,6],
    [4,0,0,6,0]
], dtype = np.float64)

s = 0
t = 4
its = 1000
burnin = 1000

samples = gibbs(A, s, t, w, its, burnin)[1]

weights = learnstcuts(A, s, t, np.transpose(samples))

print("Approximate Weights:")
print(weights)

Approximate Weights:
[[0.         0.6        0.         0.         1.        ]
 [0.6        0.         0.50472216 0.         0.        ]
 [0.         0.50472216 0.         0.69735803 0.        ]
 [0.         0.         0.69735803 0.         0.94233108]
 [1.         0.         0.         0.94233108 0.        ]]


### 2. Validation and Estimated Partition Functions

In [11]:
# Sumprod
from collections import defaultdict
from copy import deepcopy
import math
import numpy as np

def sumprod(M, s, t, w, its):
    """Sum-product algorithm

    Arguments:
        M {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        w {n x n matrix} -- weights for each edge
        its {integer} -- number of iterations

    Returns:
        The approximate partition function attained after `its` iterations by
        plugging the appropriately normalized beliefs into the Bethe free energy
    """

    n = M.shape[0]
    edges = []
    for i in range(n):
        for j in range(n):
            if M[i,j] == 1:
                edges.append((i,j))

    phi = lambda i, x_i: x_i if i == s else (1 - x_i if i == t else 1)
    psi = lambda i, j, x_i, x_j: w[i, j] ** int(x_i != x_j)

    # n x n matrix
    # Stores messages from factor nodes (C) to vertices (i).
    # C_i_messages[C, i] represents the message from the factor C
    # to the vertex i
    C_i_messages = defaultdict(lambda: 1)
    new_C_i_messages = defaultdict(lambda: 1)

    # n x n matrix
    # Stores messages from vertices (i) to factor nodes (C).
    # i_C_messages[i, C] represents the message from the vertex i
    # to the factor C
    i_C_messages = defaultdict(lambda: 1)

    for it in range(its):

        # Messages from C to i
        for (C, i) in edges:
            for x_i in range(2):
                C_i_sum = 0

                # Sum over assignments to x_C
                for x_C in range(2):
                    # Product over k in C \ i = {C}
                    # Would be a loop if I was using cliques and not just edges
                    C_i_sum += psi(C, i, x_C, x_i) * i_C_messages[C, (C, i), x_C]

                new_C_i_messages[(C, i), i, x_i] = C_i_sum

        # Messages from i to C
        for (i, C) in edges:
            for x_i in range(2):
                prod = phi(i, x_i)

                # Product over messages into i
                for (C_prime, k) in edges:
                    # Exclude C -> i
                    if k == i and C_prime != C:
                        prod *= C_i_messages[(C_prime, k), k, x_i]

                i_C_messages[i, (i, C), x_i] = prod

        C_i_messages = deepcopy(new_C_i_messages)

        # Scale
        max_val = max(map(abs, [*C_i_messages.values(), *i_C_messages.values()]))
        if max_val > 0:
            for i in C_i_messages:
                C_i_messages[i] = max(1e-50, C_i_messages[i] / max_val)
            for i in i_C_messages:
                i_C_messages[i] = max(1e-50, i_C_messages[i] / max_val)


    # Calculate beliefs for all i
    beliefs = np.empty(n)
    for i in range(n):
        belief0 = phi(i, 0)
        belief1 = phi(i, 1)

        for (C, k) in edges:
            if k == i:
                belief0 *= C_i_messages[(C, k), k, 0]
                belief1 *= C_i_messages[(C, k), k, 1]
        beliefs[i] = belief1 / (belief0 + belief1)

    # Calculate beliefs for all C
    C_beliefs = defaultdict(int)
    for (i, j) in edges:
        if (j, i) in C_beliefs:
            continue

        total = 0

        for x_i in range(2):
            for x_j in range(2):
                prob = psi(i, j, x_i, x_j)
                # Message i -> C (x_i)
                prob *= i_C_messages[i, (i, j), x_i]
                # Message j -> C (x_j)
                prob *= i_C_messages[j, (j, i), x_j]

                C_beliefs[(i, j), (x_i, x_j)] = prob
                total += prob

        # Normalize
        for x_i in range(2):
            for x_j in range(2):
                C_beliefs[(i, j), (x_i, x_j)] /= total

    bethe = 0
    eps = 1e-7

    for i in range(n):
        belief0 = 1 - beliefs[i]
        belief1 = beliefs[i]

        if belief0 > eps:
            bethe -= belief0 * math.log(belief0)

        if belief1 > eps:
            bethe -= belief1 * math.log(belief1)

    for (i, j) in edges:
        if i < j:
            for x_i in range(2):
                for x_j in range(2):
                    belief_i = beliefs[i]
                    belief_j = beliefs[j]
                    C_belief = C_beliefs[(i, j), (x_i, x_j)]

                    if x_i == 0:
                        belief_i = 1 - belief_i
                    if x_j == 0:
                        belief_j = 1 - belief_j

                    if C_belief > eps and belief_i * belief_j > 0:
                        bethe -= C_belief * math.log(C_belief / (belief_i * belief_j))

                    bethe += C_belief * math.log(psi(i, j, x_i, x_j))

    return beliefs, round(math.exp(bethe), 3)

In [12]:
import math

A = np.array([[0, 1, 0],  # 0
              [1, 0, 1],  # 1
              [0, 1, 0]], # 2
              dtype = np.float64)

w = np.array([[0, math.exp(1), 0],           # 0
              [math.exp(1), 0, math.exp(1)], # 1
              [0, math.exp(1), 0]],          # 2
              dtype = np.float64)

s = 0
t = 2
its = 10 ** 5
burnin = 10 ** 5

samples = gibbs(A, s, t, w, its, burnin)[1]

weights = learnstcuts(A, s, t, np.transpose(samples))

print(f"Approximate weights given {its} samples")
print("-" * 40)
print(f"{'':<5}" + ''.join([f"{i:^10}" for i in range(len(weights))]))
for i, row in enumerate(weights):
    print(f"{i:<5} " + " ".join(f"{val:<10.4f}" for val in row))
print()

print(f"{'Num Samples':<15} | Estimated Partition Function")
print("-" * 46)
for n in range(1, 6):
    weights = learnstcuts(A, s, t, np.transpose(samples[:10 ** n]))
    _, partition_function = sumprod(A, s, t, weights, 10 ** n)
    print(f"{10 ** n:<15} | {partition_function}")

Approximate weights given 100000 samples
----------------------------------------
         0         1         2     
0     0.0000     0.4994     0.0000    
1     0.4994     0.0000     0.5006    
2     0.0000     0.5006     0.0000    

Num Samples     | Estimated Partition Function
----------------------------------------------
10              | 1.0
100             | 1.0
1000            | 1.0
10000           | 1.0
100000          | 1.0
